# 10 · Simulate some data

**Step 0 of the workflow.** Nothing here involves NanoOrganizer — this is
just a rig writing files, the way a rig does. Run it once; `11` and `12` use
what it leaves behind.

It is deliberately **messy in the way real campaigns are messy**: three
instruments, three folder trees, three naming conventions, none of them
agreeing, and none of them laid out as `<Modality>Data/<SampleID>/`. That is
the point. A convention-based auto-attach cannot find this data, and pretending
otherwise is the thing that makes a demo lie.

```
~/Repos/OrgDemo/Lab/
├── RawData/
│   ├── spectrometer/2026-03-11/      S01_t0000s.csv …   one file per frame
│   ├── microscope_share/S01/         img_01.tif …       a folder per sample
│   └── xrd_rig/                      S01_waxs.dat       one file per sample
└── Meta/
    └── synthesis_dict.json           what the operator wrote down
```

Only the spectra get written into the metadata dict. The micrographs and the
diffraction come from other people on other days, and in `11` they are linked
by hand — which is what actually happens.

**The hidden control variable is the synthesis temperature.** It sets the
particle size, which sets the plasmon band, the diffraction line width and
what the micrographs show. `12` tries to recover it.

In [1]:
from pathlib import Path
import json

import numpy as np

from NanoOrganizer.demo import demo_root

LAB = demo_root("Lab")
RAW = LAB / "RawData"
SPECTRA = RAW / "spectrometer" / "2026-03-11"
SCOPE = RAW / "microscope_share"
XRD = RAW / "xrd_rig"
META = LAB / "Meta"

for folder in (SPECTRA, SCOPE, XRD, META):
    folder.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(7)

# One hidden number per sample. Everything below follows from it.
TEMPERATURE_C = {"S01": 60.0, "S02": 70.0, "S03": 80.0,
                 "S04": 90.0, "S05": 100.0, "S06": 110.0}

def diameter_nm(temperature_C):
    """Hotter synthesis, fewer nuclei, bigger particles."""
    return 6.0 + 0.16 * (temperature_C - 60.0)

def band_nm(d):
    """Plasmon band red-shifts as the particle grows."""
    return 512.0 + 1.9 * d

print({s: round(diameter_nm(t), 1) for s, t in TEMPERATURE_C.items()})

{'S01': 6.0, 'S02': 7.6, 'S03': 9.2, 'S04': 10.8, 'S05': 12.4, 'S06': 14.0}


## The spectrometer

A growth series per sample: fourteen frames over twenty minutes, each a
two-column CSV, with the acquisition time in the filename. The band grows and
red-shifts as the particles do.

In [2]:
WAVELENGTH = np.linspace(380.0, 780.0, 401)
N_FRAMES = 14

for sample, temperature in TEMPERATURE_C.items():
    final_d = diameter_nm(temperature)
    for frame in range(N_FRAMES):
        progress = (frame + 1) / N_FRAMES
        # Growth saturates, so the band moves fast early and then settles.
        d = final_d * (1.0 - np.exp(-3.0 * progress))
        centre = band_nm(d)
        width = 44.0 - 0.9 * d
        signal = (0.08 + 0.9 * progress) * np.exp(
            -0.5 * ((WAVELENGTH - centre) / width) ** 2)
        baseline = 0.04 + 8e3 / WAVELENGTH ** 2        # scattering background
        noise = rng.normal(0.0, 0.004, WAVELENGTH.size)

        seconds = frame * 90
        path = SPECTRA / f"{sample}_t{seconds:04d}s.csv"
        np.savetxt(path, np.column_stack([WAVELENGTH, signal + baseline + noise]),
                   delimiter=",", header="wavelength_nm,absorbance", comments="")

print(len(list(SPECTRA.glob("*.csv"))), "spectra in", SPECTRA)
print(sorted(p.name for p in SPECTRA.glob("S01_*.csv"))[:4])

84 spectra in /home/yuzhang/Repos/OrgDemo/Lab/RawData/spectrometer/2026-03-11
['S01_t0000s.csv', 'S01_t0090s.csv', 'S01_t0180s.csv', 'S01_t0270s.csv']


## The microscope

A folder per sample of calibrated TIFFs, written by whoever was on the
instrument that week. The pixel calibration goes in the image description, so
the micrographs can be read in nanometres later without anyone remembering the
magnification.

In [3]:
from PIL import Image

NM_PER_PIXEL = 0.5
SIZE = 320

def micrograph(d_nm):
    """Dark particles of diameter *d_nm* on a noisy support."""
    field = np.full((SIZE, SIZE), 196.0) + rng.normal(0, 5, (SIZE, SIZE))
    radius_px = 0.5 * d_nm / NM_PER_PIXEL
    yy, xx = np.ogrid[:SIZE, :SIZE]
    for _ in range(26):
        cy, cx = rng.uniform(radius_px, SIZE - radius_px, 2)
        r = radius_px * rng.normal(1.0, 0.12)
        mask = (yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2
        field[mask] = 70.0 + rng.normal(0, 4)
    return np.clip(field, 0, 255)

pixels_per_micron = 1000.0 / NM_PER_PIXEL
description = (f"LabScope XpixCal={pixels_per_micron:.6f}"
               f"YpixCal={pixels_per_micron:.6f}Unit=um")

for sample, temperature in TEMPERATURE_C.items():
    folder = SCOPE / sample
    folder.mkdir(exist_ok=True)
    for index in range(1, 4):
        array = micrograph(diameter_nm(temperature)).astype(np.uint8)
        Image.fromarray(array).save(folder / f"img_{index:02d}.tif",
                                    description=description)
    (folder / "session.txt").write_text(
        f"{sample}: 3 fields, 0.5 nm/px, operator RH\n")

print(sorted(p.name for p in (SCOPE / "S01").iterdir()))

['img_01.tif', 'img_02.tif', 'img_03.tif', 'session.txt']


## The diffractometer

One pattern per sample. The line width narrows as the crystallites grow —
Scherrer — so diffraction sees the same hidden number the spectra do, from a
completely different direction.

In [4]:
Q = np.linspace(2.0, 5.0, 900)
PEAKS = ((2.67, 1.00), (3.08, 0.46), (4.36, 0.22))      # fcc 111, 200, 220

for sample, temperature in TEMPERATURE_C.items():
    d = diameter_nm(temperature)
    fwhm = 0.9 * 2 * np.pi / (d * 10.0)                 # Scherrer, A^-1
    sigma = fwhm / 2.355
    pattern = 60.0 / Q                                   # background
    for centre, height in PEAKS:
        pattern += 900 * height * np.exp(-0.5 * ((Q - centre) / sigma) ** 2)
    pattern += rng.normal(0, 4, Q.size)

    np.savetxt(XRD / f"{sample}_waxs.dat", np.column_stack([Q, pattern]),
               header="q_invA  intensity", comments="# ")

print(sorted(p.name for p in XRD.iterdir()))

['S01_waxs.dat', 'S02_waxs.dat', 'S03_waxs.dat', 'S04_waxs.dat', 'S05_waxs.dat', 'S06_waxs.dat']


## What the operator wrote down

The metadata dict: one record per sample, keyed by sample id. It holds the
synthesis conditions and **the spectra only** — the block naming files becomes
a measurement when it is ingested.

The micrographs and the diffraction are *not* in here. Nobody wrote them down,
which is the normal state of affairs and the reason `11` links them by hand.

Saved as JSON so `11` can load the real dict rather than re-running this
notebook.

In [5]:
Synthesis_dict = {}
for index, (sample, temperature) in enumerate(TEMPERATURE_C.items(), start=1):
    Synthesis_dict[sample] = {
        "sample_id": sample,
        "synthesis_batch": {
            "run_id": f"run_{index:03d}",
            "batch_tag": "2026-03-11",
            "campaign": "lab-demo",
            "status": "done",
            "started_at": f"2026-03-11T09:{index * 7:02d}:00",
        },
        "conditions": {
            "temperature_C": temperature,
            "hold_time_min": 21.0,
            "stir_rate_rpm": 400,
            "solvent": "water",
        },
        "reagents": {
            "precursor": {"concentration_mM": 2.0, "volume_uL": 500.0},
            "reductant": {"concentration_mM": 16.0, "volume_uL": 250.0},
        },
        "uvvis_growth": {
            "modality": "uvvis",
            "instrument": "LabSpec 2000",
            "n_frames": N_FRAMES,
            "spectrum_glob": f"{SPECTRA}/{sample}_t*.csv",
        },
    }

(META / "synthesis_dict.json").write_text(json.dumps(Synthesis_dict, indent=2))
print("wrote", META / "synthesis_dict.json")
Synthesis_dict["S01"]

wrote /home/yuzhang/Repos/OrgDemo/Lab/Meta/synthesis_dict.json


{'sample_id': 'S01',
 'synthesis_batch': {'run_id': 'run_001',
  'batch_tag': '2026-03-11',
  'campaign': 'lab-demo',
  'status': 'done',
  'started_at': '2026-03-11T09:07:00'},
 'conditions': {'temperature_C': 60.0,
  'hold_time_min': 21.0,
  'stir_rate_rpm': 400,
  'solvent': 'water'},
 'reagents': {'precursor': {'concentration_mM': 2.0, 'volume_uL': 500.0},
  'reductant': {'concentration_mM': 16.0, 'volume_uL': 250.0}},
 'uvvis_growth': {'modality': 'uvvis',
  'instrument': 'LabSpec 2000',
  'n_frames': 14,
  'spectrum_glob': '/home/yuzhang/Repos/OrgDemo/Lab/RawData/spectrometer/2026-03-11/S01_t*.csv'}}

## The answer key

Kept beside the data so `12` can check what the pipeline recovers against what
the generator actually used — the one thing real data never lets you do.

In [6]:
import pandas as pd

truth = pd.DataFrame([
    {"sample_id": s,
     "temperature_C": t,
     "true_diameter_nm": round(diameter_nm(t), 2),
     "true_band_nm": round(band_nm(diameter_nm(t)), 1)}
    for s, t in TEMPERATURE_C.items()
])
truth.to_csv(LAB / "Meta" / "truth.csv", index=False)
truth

,sample_id,temperature_C,true_diameter_nm,true_band_nm
0,S01,60.0,6.0,523.4
1,S02,70.0,7.6,526.4
2,S03,80.0,9.2,529.5
3,S04,90.0,10.8,532.5
4,S05,100.0,12.4,535.6
5,S06,110.0,14.0,538.6


---

The data is on disk and nothing knows about it yet. **Next: `11_build_organizer`.**